# Retail Sales - Exploratory Data Analysis
**Oasis Infobyte Data Analytics Internship | Level 1 | EDA Retail Sales**

---


## Project Introduction
This project performs a comprehensive Exploratory Data Analysis (EDA) on a retail sales dataset. The goal is to extract actionable business insights through statistical analysis, data visualization, and pattern recognition.

The analysis is structured into systematic stages: data ingestion, cleaning, feature engineering, univariate and multivariate analysis, time-series evaluation, and business interpretation.


## Business Objective
The core business objectives are:
1. Identify top-performing product categories and sub-categories
2. Analyze regional and segment-wise sales performance
3. Understand the relationship between discounts and profitability
4. Detect seasonal trends and order patterns
5. Recognize high-value customers for targeted marketing
6. Provide data-driven recommendations to improve revenue and profit margins


## Dataset Description
The dataset contains **5,075 retail transaction records** with 13 attributes:

| Column | Type | Description |
|--------|------|-------------|
| Order ID | String | Unique identifier for each order |
| Order Date | DateTime | Date when the order was placed |
| Ship Date | DateTime | Date when the order was shipped |
| Customer Name | String | Name of the customer |
| Segment | Categorical | Customer segment (Consumer, Corporate, Home Office) |
| Region | Categorical | Geographic region (North, South, East, West) |
| State | Categorical | U.S. state |
| Category | Categorical | Product category (Furniture, Office Supplies, Technology) |
| Sub-Category | Categorical | Product sub-category |
| Sales | Numerical | Total sales amount in USD |
| Quantity | Numerical | Number of units sold |
| Discount | Numerical | Discount percentage (0.0 to 0.5) |
| Profit | Numerical | Profit earned in USD |

**Data Quality Issues Introduced:**
- ~3% missing values in `Profit`
- ~2% missing values in `Discount`
- ~1.5% duplicate rows


## Import Libraries


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from datetime import datetime
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 150
plt.rcParams['savefig.dpi'] = 150
plt.rcParams['font.size'] = 10

OUTPUT_DIR = 'outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Libraries imported successfully.')


## Load Dataset


In [ ]:
file_path = 'data/retail_sales.csv'
df = pd.read_csv(file_path, parse_dates=['Order Date', 'Ship Date'])
print(f'Dataset loaded successfully. Shape: {df.shape}')


## Data Inspection
Let's examine the first few rows, column types, and basic statistics.


In [ ]:
print('=== FIRST 5 ROWS ===')
display(df.head())
print('\n=== DATA INFO ===')
print(df.info())
print('\n=== COLUMNS ===')
print(df.columns.tolist())


## Data Cleaning
### Step 1: Remove Duplicates


In [ ]:
print(f'Duplicates before removal: {df.duplicated().sum()}')
df = df.drop_duplicates()
print(f'Duplicates after removal: {df.duplicated().sum()}')


## Handle Missing Values


In [ ]:
print('=== MISSING VALUES BEFORE CLEANING ===')
print(df.isnull().sum())


In [ ]:
num_cols = ['Sales', 'Quantity', 'Discount', 'Profit']
for col in num_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].median())
        print(f'Filled missing values in {col} with median.')

cat_cols = ['Segment', 'Region', 'State', 'Category', 'Sub-Category', 'Customer Name']
for col in cat_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna('Unknown')
        print(f'Filled missing values in {col} with Unknown.')


In [ ]:
print('=== MISSING VALUES AFTER CLEANING ===')
print(df.isnull().sum())


## Convert Date Columns & Feature Engineering


In [ ]:
df['Order Year'] = df['Order Date'].dt.year
df['Order Month'] = df['Order Date'].dt.to_period('M').astype(str)
df['Order Month Name'] = df['Order Date'].dt.strftime('%b-%Y')
df['Profit Margin %'] = np.where(df['Sales'] != 0, (df['Profit'] / df['Sales']) * 100, 0)
df['Sales per Unit'] = np.where(df['Quantity'] != 0, df['Sales'] / df['Quantity'], 0)

print('Feature engineering complete.')
print(f'New columns added: Order Year, Order Month, Order Month Name, Profit Margin %, Sales per Unit')


## Descriptive Statistics


In [ ]:
print('=== NUMERICAL FEATURES SUMMARY ===')
display(df[['Sales', 'Quantity', 'Discount', 'Profit']].describe())


In [ ]:
print('=== CATEGORICAL FEATURES SUMMARY ===')
for col in ['Segment', 'Region', 'Category', 'Sub-Category']:
    print(f'\n{col}:')
    print(df[col].value_counts().head())


## Univariate Analysis
### Sales Distribution


**What is happening:** The histogram shows that most sales transactions fall between $50 and $500, with a long right tail indicating occasional high-value orders.

**Why it matters:** Understanding the distribution helps set pricing strategies and identify outlier deals.

**Business impact:** High-value outliers may represent bulk or corporate orders that require special handling.

**Recommendation:** Investigate high-value transactions for upsell opportunities and maintain service quality for bulk orders.


In [ ]:
def save_plot(fig, name):
    path = os.path.join(OUTPUT_DIR, name)
    fig.savefig(path, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    print(f'[OK] Saved: {path}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df['Sales'], bins=50, kde=True, color='#2E86AB', ax=axes[0])
axes[0].set_title('Sales Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Sales ($)', fontsize=11)
axes[0].set_ylabel('Frequency', fontsize=11)
axes[0].grid(True, alpha=0.3)
sns.boxplot(x=df['Sales'], color='#A23B72', ax=axes[1])
axes[1].set_title('Sales Box Plot (Outliers)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Sales ($)', fontsize=11)
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
save_plot(fig, 'sales_distribution.png')


### Profit Distribution


**What is happening:** The profit distribution is centered around positive values but has a left tail showing loss-making transactions.

**Why it matters:** Profitability is the ultimate business metric. Identifying loss patterns is critical for margin protection.

**Business impact:** Even a small percentage of unprofitable orders can erode overall margins significantly.

**Recommendation:** Implement profit margin thresholds for order approval and review discount policies regularly.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df['Profit'], bins=50, kde=True, color='#F18F01', ax=axes[0])
axes[0].set_title('Profit Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Profit ($)', fontsize=11)
axes[0].set_ylabel('Frequency', fontsize=11)
axes[0].grid(True, alpha=0.3)
sns.boxplot(x=df['Profit'], color='#C73E1D', ax=axes[1])
axes[1].set_title('Profit Box Plot', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Profit ($)', fontsize=11)
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
save_plot(fig, 'profit_distribution.png')


## Time Series Analysis
### Monthly Sales & Profit Trends


**What is happening:** Sales exhibit a clear seasonal pattern with peaks in Q4 (October-December) and troughs in Q1 and Q2.

**Why it matters:** Inventory planning, marketing budgets, and staffing should align with seasonal demand cycles.

**Business impact:** Missing the seasonal peak can result in stockouts and lost revenue, while overstocking in off-peak months increases carrying costs.

**Recommendation:** Increase inventory levels by 20-30% before Q4 and launch promotional campaigns in Q1 to smooth demand.


In [ ]:
monthly = df.groupby('Order Month').agg({'Sales': 'sum', 'Profit': 'sum'}).reset_index()
monthly['Order Month'] = pd.to_datetime(monthly['Order Month'])
monthly = monthly.sort_values('Order Month')

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
axes[0].plot(monthly['Order Month'], monthly['Sales'], marker='o', color='#2E86AB', linewidth=2)
axes[0].fill_between(monthly['Order Month'], monthly['Sales'], alpha=0.3, color='#2E86AB')
axes[0].set_title('Monthly Sales Trend', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Total Sales ($)', fontsize=11)
axes[0].grid(True, alpha=0.3)
axes[1].plot(monthly['Order Month'], monthly['Profit'], marker='s', color='#28A745', linewidth=2)
axes[1].fill_between(monthly['Order Month'], monthly['Profit'], alpha=0.3, color='#28A745')
axes[1].set_title('Monthly Profit Trend', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Month', fontsize=11)
axes[1].set_ylabel('Total Profit ($)', fontsize=11)
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
save_plot(fig, 'monthly_sales_trend.png')


## Category Analysis
### Sales by Category


**What is happening:** Technology is the highest revenue generator, followed by Furniture and Office Supplies.

**Why it matters:** Category performance directly impacts inventory decisions and marketing allocation.

**Business impact:** Technology products may have higher margins but lower volume, while Furniture drives volume with moderate margins.

**Recommendation:** Expand Technology product range and bundle Furniture with accessories to increase average order value.


In [ ]:
cat_sales = df.groupby('Category')['Sales'].sum().sort_values(ascending=False).reset_index()
colors = ['#2E86AB', '#A23B72', '#F18F01']
fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(cat_sales['Category'], cat_sales['Sales'], color=colors, edgecolor='black', linewidth=0.5)
ax.set_title('Sales by Category', fontsize=13, fontweight='bold')
ax.set_xlabel('Category', fontsize=11)
ax.set_ylabel('Total Sales ($)', fontsize=11)
ax.grid(True, axis='y', alpha=0.3)
for bar in bars:
    height = bar.get_height()
    ax.annotate(f'${height:,.0f}', xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3), textcoords='offset points', ha='center', va='bottom', fontsize=10)
plt.tight_layout()
save_plot(fig, 'category_sales.png')


### Sales by Sub-Category


**What is happening:** Phones and Chairs are the top sub-categories, while Labels and Fasteners contribute the least.

**Why it matters:** Sub-category analysis reveals micro-level product opportunities and gaps.

**Business impact:** Low-performing sub-categories may indicate poor visibility, pricing issues, or lack of demand.

**Recommendation:** Consider phasing out low-margin sub-categories and investing in top performers with expanded SKUs.


In [ ]:
subcat_sales = df.groupby('Sub-Category')['Sales'].sum().sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(12, 7))
colors = plt.cm.viridis(np.linspace(0, 1, len(subcat_sales)))
bars = ax.barh(subcat_sales.index, subcat_sales.values, color=colors, edgecolor='black', linewidth=0.3)
ax.set_title('Sales by Sub-Category', fontsize=13, fontweight='bold')
ax.set_xlabel('Total Sales ($)', fontsize=11)
ax.set_ylabel('Sub-Category', fontsize=11)
ax.grid(True, axis='x', alpha=0.3)
for bar in bars:
    width = bar.get_width()
    ax.annotate(f'${width:,.0f}', xy=(width, bar.get_y() + bar.get_height() / 2),
                xytext=(5, 0), textcoords='offset points', va='center', fontsize=9)
plt.tight_layout()
save_plot(fig, 'subcategory_sales.png')


## Region Analysis
### Sales by Region


**What is happening:** The West region leads in sales share, followed by East, South, and North.

**Why it matters:** Regional performance differences highlight market penetration levels and economic factors.

**Business impact:** Underperforming regions may need localized marketing, distribution network improvements, or pricing adjustments.

**Recommendation:** Allocate 40% of marketing budget to West and East, 35% to South (growth market), and 25% to North (maintenance).


In [ ]:
reg_sales = df.groupby('Region')['Sales'].sum().sort_values(ascending=False).reset_index()
colors = ['#2E86AB', '#A23B72', '#F18F01', '#28A745']
fig, ax = plt.subplots(figsize=(10, 6))
wedges, texts, autotexts = ax.pie(reg_sales['Sales'], labels=reg_sales['Region'], autopct='%1.1f%%',
                                   colors=colors, startangle=90, explode=(0.03, 0.03, 0.03, 0.03),
                                   textprops={'fontsize': 11})
ax.set_title('Sales by Region', fontsize=13, fontweight='bold')
plt.tight_layout()
save_plot(fig, 'regional_sales.png')


### Sales by State


**What is happening:** California and New York dominate state-level sales, with Texas and Florida as emerging markets.

**Why it matters:** State-level insights help optimize logistics, warehousing, and regional sales teams.

**Business impact:** Concentrating resources on top states maximizes ROI on sales and marketing spend.

**Recommendation:** Establish regional distribution hubs in California and New York, and pilot expansion in Texas and Florida.


In [ ]:
state_sales = df.groupby('State')['Sales'].sum().sort_values(ascending=False).head(10).reset_index()
fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(data=state_sales, x='State', y='Sales', palette='Blues_r', ax=ax)
ax.set_title('Top 10 States by Sales', fontsize=13, fontweight='bold')
ax.set_xlabel('State', fontsize=11)
ax.set_ylabel('Total Sales ($)', fontsize=11)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
ax.grid(True, axis='y', alpha=0.3)
for p in ax.patches:
    ax.annotate(f'${p.get_height():,.0f}', (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontsize=9)
plt.tight_layout()
save_plot(fig, 'state_sales.png')


## Customer Analysis
### Top 10 Customers


**What is happening:** A small group of customers generates a large portion of total revenue (Pareto principle).

**Why it matters:** Retaining high-value customers is far more cost-effective than acquiring new ones.

**Business impact:** Losing a top customer can have a material impact on quarterly revenue.

**Recommendation:** Implement VIP loyalty programs, dedicated account managers, and exclusive early-access offers for top customers.


In [ ]:
top_cust = df.groupby('Customer Name')['Sales'].sum().sort_values(ascending=False).head(10).reset_index()
fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(data=top_cust, x='Sales', y='Customer Name', palette='rocket', ax=ax)
ax.set_title('Top 10 Customers by Sales', fontsize=13, fontweight='bold')
ax.set_xlabel('Total Sales ($)', fontsize=11)
ax.set_ylabel('Customer Name', fontsize=11)
ax.grid(True, axis='x', alpha=0.3)
for p in ax.patches:
    ax.annotate(f'${p.get_width():,.0f}', (p.get_width(), p.get_y() + p.get_height() / 2),
                ha='left', va='center', fontsize=9)
plt.tight_layout()
save_plot(fig, 'top_customers.png')


## Bivariate Analysis
### Discount vs Profit


**What is happening:** There is a clear negative relationship between discount and profit. As discounts increase beyond 25%, profits drop sharply, with many orders becoming loss-making.

**Why it matters:** Discounting is a common sales lever, but it can destroy profitability if not controlled.

**Business impact:** Unchecked discounting can turn a profitable business into a loss-making one.

**Recommendation:** Set a maximum discount threshold of 25%. Require manager approval for discounts above 15%. Monitor discount-to-profit ratio weekly.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sns.regplot(data=df, x='Discount', y='Profit', scatter_kws={'alpha': 0.4, 'color': '#A23B72'},
            line_kws={'color': '#C73E1D', 'linewidth': 2}, ax=ax)
ax.set_title('Discount vs Profit', fontsize=13, fontweight='bold')
ax.set_xlabel('Discount', fontsize=11)
ax.set_ylabel('Profit ($)', fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
save_plot(fig, 'discount_vs_profit.png')


## Correlation Analysis


**What is happening:** Sales and Profit show a strong positive correlation. Discount shows a strong negative correlation with Profit Margin %.

**Why it matters:** Correlation analysis validates business assumptions and reveals hidden relationships.

**Business impact:** Understanding these relationships helps prioritize which levers (pricing, volume, discounts) to pull for growth.

**Recommendation:** Focus on driving sales volume through marketing rather than discounting to protect margins.


In [ ]:
corr_cols = ['Sales', 'Quantity', 'Discount', 'Profit', 'Profit Margin %', 'Sales per Unit']
corr = df[corr_cols].corr()
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, fmt='.2f',
            linewidths=0.5, ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('Correlation Heatmap', fontsize=13, fontweight='bold')
plt.tight_layout()
save_plot(fig, 'correlation_heatmap.png')


## Multivariate Analysis
### Pairplot of Key Numerical Features


**What is happening:** The pairplot reveals natural clustering between Sales and Profit, while Discount shows inverse behavior with Profit.

**Why it matters:** Multivariate visualization helps identify complex patterns that univariate analysis might miss.

**Business impact:** Clustering can inform customer segmentation and personalized marketing strategies.

**Recommendation:** Use clustering insights to develop tiered pricing and product bundles for different customer segments.


In [ ]:
pair_cols = ['Sales', 'Quantity', 'Discount', 'Profit']
fig = sns.pairplot(df[pair_cols], diag_kind='kde', plot_kws={'alpha': 0.5, 's': 20})
fig.fig.suptitle('Pairplot of Key Numerical Features', y=1.02, fontsize=14, fontweight='bold')
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, 'pairplot.png'), bbox_inches='tight', facecolor='white')
plt.close('all')
print('[OK] Saved: outputs/pairplot.png')


## Business Insights Summary
### Key Findings

1. **Seasonality is Real:** Q4 consistently outperforms other quarters. Plan inventory and staffing accordingly.
2. **Technology is King:** Highest revenue generator with strong margins. Expand this category.
3. **Discounts are Dangerous:** >30% discounts lead to losses. Cap at 25%.
4. **West Region Leads:** Prioritize resource allocation to West and East markets.
5. **Top Customers Drive Revenue:** A small customer base contributes disproportionately to revenue.
6. **Furniture Volume vs Margin:** High sales volume but lower margins due to discounting and shipping costs.

### Recommendations

| Priority | Action | Expected Impact |
|----------|--------|-----------------|
| High | Cap discounts at 25% | Improve profit margins by 8-12% |
| High | Increase Q4 inventory | Reduce stockouts, capture seasonal demand |
| Medium | Expand Technology SKUs | Increase average order value |
| Medium | Launch loyalty program for top 10% customers | Improve retention by 15-20% |
| Low | Phase out low-margin sub-categories | Reduce operational complexity |


## Final Conclusion
This EDA reveals a retail business with strong seasonal patterns, clear category leaders, and significant opportunities in discount management and customer retention. The data suggests that focusing on Technology products, controlling discounting, and nurturing high-value customers will drive the most significant improvements in revenue and profitability. The West and East regions present the best growth opportunities, while the South region shows untapped potential. By implementing the recommended strategies, the business can expect improved margins, higher customer lifetime value, and sustainable growth.
